# Fine-Tune a Small LLM for Extraction, with LoRA

In the extraction lab, a 4-billion-parameter LLM extracted the entities of the 1914 press. It also labelled 1,999 other passages: these labels will teach the task to a model **six times smaller**, Qwen3-0.6B, with LoRA. That's distillation: the large model serves once, the small one serves in production.

1. The small model as is: how good is it?
2. Prepare the training data
3. LoRA: how many parameters do we train?
4. Train
5. The fine-tuned small model against the large one

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs, on a smaller scale: the extractions cover 4 passages instead of 40, the training 2 examples, and exercise 5 loads the adapters trained on a GPU.

## Data and Model

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/gold.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/distill.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/press-1914-extraction/teacher-gold.jsonl
!wget -q https://github.com/shuuchuu/models/raw/refs/heads/main/press-extraction-lora-qwen3-0.6b/en/adapter_model.safetensors
%pip install -q peft trl rapidfuzz
%pip uninstall -q -y torchao  # Colab's version is too old for peft

In [ ]:
import json
import time

import pandas as pd
import torch
from rapidfuzz import fuzz
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# On a CPU, float32: half precisions are very slow there.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "No GPU: the long computations will load their result")

MODEL = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map=DEVICE)

gold = pd.read_json("gold.jsonl", lines=True)
distill = pd.read_json("distill.jsonl", lines=True)
teacher = pd.read_json("teacher-gold.jsonl", lines=True)
# The test passages of exercises 1 and 5: the 40 on a GPU; without one, 4 (about three
# minutes per extraction).
gold_part = gold[: None if GPU else 4]
FIELDS = ["people", "places", "organizations"]
print(len(gold), "test passages,", len(distill), "passages labelled by the large model")

The extraction lab's measure (F1 per field, lenient matching), and the prompt (the one given to the large model, translated).

In [ ]:
def normalize(name: str) -> str:
  return " ".join(name.lower().replace("’", "'").split())


def same(a: str, b: str) -> bool:
  a, b = normalize(a), normalize(b)
  return fuzz.ratio(a, b) >= 85 or (min(len(a), len(b)) >= 4 and (a in b or b in a))


def evaluate(predictions: list[dict[str, list[str]]]) -> pd.Series:
  scores = {}
  for field in FIELDS:
    found = expected = right_found = right_expected = 0
    for prediction, (_, row) in zip(predictions, gold.iterrows()):
      value = prediction.get(field)  # a small model can answer anything
      predicted = (
        list(dict.fromkeys(p for p in value if isinstance(p, str)))
        if isinstance(value, list)
        else []
      )
      truth = row[field]
      found += len(predicted)
      expected += len(truth)
      right_found += sum(any(same(p, t) for t in truth) for p in predicted)
      right_expected += sum(any(same(p, t) for p in predicted) for t in truth)
    precision, recall = right_found / max(found, 1), right_expected / max(expected, 1)
    scores[field] = 2 * precision * recall / max(precision + recall, 1e-9)
  return pd.Series(scores).round(2)


INSTRUCTIONS = """Extract the named entities of this passage of the 1914 French press.
- people: persons, without their title (M., Mme, sir, lord);
- places: places (countries, regions, towns, streets, buildings);
- organizations: named organizations (assemblies, parties, unions, newspapers, agencies).
Each entity once, as written in the text, errors included.
No generic mentions such as "le gouvernement". Do not invent anything.
Answer only with a JSON object with the keys people, places and organizations."""


def messages(text: str) -> list[dict[str, str]]:
  return [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": text},
  ]


print("Large model:", evaluate(teacher.to_dict("records")).to_dict())

## 1. The Small Model as Is

Write `extract_all(texts)` that generates, in batches of 8, the model's answer to each passage (`tokenizer.apply_chat_template`, then `model.generate`), and reads it as JSON (`json.loads`, an empty dictionary if the answer isn't valid JSON). Measure the F1 on the test passages of `gold_part` (the 40; without a GPU, the first 4), the share of invalid answers and the time per passage.

**Useful documentation**

- [Chat templates](https://huggingface.co/docs/transformers/chat_templating)
- [`Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`model.generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)
- [`tokenizer.decode`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.decode)
- [`json.loads`](https://docs.python.org/3/library/json.html#json.loads)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `tokenizer.apply_chat_template(messages(t), tokenize=False, add_generation_prompt=True, enable_thinking=False)`: the prompt as text, ending with the start of the assistant's answer; `enable_thinking=False`: Qwen3 answers without a thinking phase
- `tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)`: a batch of prompts (left padding is already set)
- `model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None, top_p=None, top_k=None)`, under `torch.no_grad()`: returns the prompt followed by the answer, so keep the tokens after `inputs.input_ids.shape[1]`; the three `None` remove Qwen3's sampling settings, which do nothing here and would make transformers print a warning
- `tokenizer.decode(output, skip_special_tokens=True).strip()`; remove the prefix `` ```json `` and the suffix `` ``` `` the model sometimes adds (`removeprefix`, `removesuffix`)
- `json.loads(answer)` raises `json.JSONDecodeError` on invalid JSON: append `{}` and count an invalid answer (likewise if the JSON isn't a dictionary)
- the time per passage: `(time.time() - start) / len(gold_part)`

A skeleton to copy into the exercise's cell, replacing each `...`:

````python
def extract_all(texts: list[str], batch_size: int = 8) -> tuple[list[dict], int]:
  results, invalid = [], 0
  for start in range(0, len(texts), batch_size):
    prompts = [
      tokenizer.apply_chat_template(
        messages(t), tokenize=False, add_generation_prompt=True, enable_thinking=...
      )
      for t in texts[start : start + batch_size]
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=...,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    for output in outputs[:, inputs.input_ids.shape[1] :]:
      answer = tokenizer.decode(output, skip_special_tokens=True).strip()
      answer = answer.removeprefix("```json").removesuffix("```").strip()
      try:
        parsed = json.loads(...)
        results.append(parsed if isinstance(parsed, dict) else {})
      except json.JSONDecodeError:
        results.append({})
        invalid += 1
  return results, invalid


start = time.time()
base_predictions, invalid = extract_all(gold_part.text.tolist())
base_seconds = (time.time() - start) / len(gold_part)
print(f"{invalid} invalid answers, {base_seconds:.2f} s per passage")
evaluate(base_predictions)
````

### Solution

In [ ]:
def extract_all(texts: list[str], batch_size: int = 8) -> tuple[list[dict], int]:
  results, invalid = [], 0
  for start in range(0, len(texts), batch_size):
    prompts = [
      tokenizer.apply_chat_template(
        messages(t), tokenize=False, add_generation_prompt=True, enable_thinking=False
      )
      for t in texts[start : start + batch_size]
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
      outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
      )
    for output in outputs[:, inputs.input_ids.shape[1] :]:
      answer = tokenizer.decode(output, skip_special_tokens=True).strip()
      answer = answer.removeprefix("```json").removesuffix("```").strip()
      try:
        parsed = json.loads(answer)
        results.append(parsed if isinstance(parsed, dict) else {})
      except json.JSONDecodeError:
        results.append({})
        invalid += 1
  return results, invalid


start = time.time()
base_predictions, invalid = extract_all(gold_part.text.tolist())
base_seconds = (time.time() - start) / len(gold_part)
print(f"{invalid} invalid answers, {base_seconds:.2f} s per passage")
evaluate(base_predictions)

## 2. Prepare the Training Data

Each training example is a conversation: the prompt (instructions and passage), and the expected answer, the large model's JSON object. Build a `datasets.Dataset` with the columns `prompt` (the list of messages) and `completion` (a list of one `assistant` message, whose content is the labels as JSON (`json.dumps`)). Keep 100 examples for validation. Print the first training example and look at it: the large model's labels aren't perfect (in the solution's one, titles kept, the same people also filed under organizations, and a "rue Clichy" absent from the text, which says "boulevard de Clichy"), and the small model will learn their mistakes too.

**Useful documentation**

- [TRL's dataset formats](https://huggingface.co/docs/trl/dataset_formats#prompt-completion)
- [`Dataset.from_list`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.from_list)
- [`Dataset.train_test_split`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.train_test_split)
- [`json.dumps`](https://docs.python.org/3/library/json.html#json.dumps)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `distill.to_dict("records")`: a list of dictionaries, one per passage (`row["text"]`, and one field per entry of `FIELDS`)
- `messages(row["text"])`: the prompt, already a list of messages; `{field: row[field] for field in FIELDS}`: the labels
- `{"role": "assistant", "content": json.dumps(labels, ensure_ascii=False)}`: `ensure_ascii=False` keeps accented letters as they are
- `Dataset.from_list(examples)`, then `.train_test_split(test_size=100, seed=0)`: an integer is a number of examples, not a proportion

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from datasets import Dataset


def example(row) -> dict:
  labels = {field: row[field] for field in FIELDS}
  return {
    "prompt": messages(row["text"]),
    "completion": [
      {"role": "assistant", "content": json.dumps(labels, ensure_ascii=...)}
    ],
  }


dataset = Dataset.from_list([example(row) for row in distill.to_dict("records")])
split = dataset.train_test_split(test_size=..., seed=0)
print(split)
print(split["train"][0]["completion"][0]["content"])
```

### Solution

In [ ]:
from datasets import Dataset


def example(row) -> dict:
  labels = {field: row[field] for field in FIELDS}
  return {
    "prompt": messages(row["text"]),
    "completion": [
      {"role": "assistant", "content": json.dumps(labels, ensure_ascii=False)}
    ],
  }


dataset = Dataset.from_list([example(row) for row in distill.to_dict("records")])
split = dataset.train_test_split(test_size=100, seed=0)
print(split)
print(split["train"][0]["completion"][0]["content"])

## 3. LoRA

A rank-$r$ adapter on a linear layer with $d_{in}$ inputs and $d_{out}$ outputs adds $A$ ($r \times d_{in}$) and $B$ ($d_{out} \times r$): $r (d_{in} + d_{out})$ parameters.

First compute, from `model.config`, what rank-16 adapters cost on the four attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), per layer then over the whole model. Then add them to the model with `peft` (`LoraConfig`), and check your computation: how many parameters are trained, out of how many?

Common practice adapts all the linear layers, attention and MLP; this lab stays on the attention, like the adapters loaded in exercise 5. Measure, without training, what `target_modules="all-linear"` would cost.

**Useful documentation**

- [`Qwen3Config`](https://huggingface.co/docs/transformers/model_doc/qwen3#transformers.Qwen3Config)
- [`LoraConfig`](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig)
- [`get_peft_model`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.get_peft_model)
- [`PeftModel.print_trainable_parameters`](https://huggingface.co/docs/peft/package_reference/peft_model#peft.PeftModel.print_trainable_parameters)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `model.config`: `hidden_size`, `num_attention_heads` (the heads of `q_proj`), `num_key_value_heads` (those of `k_proj` and `v_proj`, fewer), `head_dim` (the width of a head), `num_hidden_layers`
- `q_proj` goes from `hidden_size` to `num_attention_heads * head_dim`, `k_proj` and `v_proj` from `hidden_size` to `num_key_value_heads * head_dim`, `o_proj` from `num_attention_heads * head_dim` to `hidden_size`; `print(model.model.layers[0].self_attn)` shows these dimensions (`in_features`, `out_features`), to check
- `LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")`: `r` is the adapters' rank, `target_modules` the layers that get them; `target_modules="all-linear"` takes them all, except the output one
- `get_peft_model(model, config)` returns the model with the adapters: keep it in `model`. It changes the model it receives: for `all-linear`, load another copy (`AutoModelForCausalLM.from_pretrained`, as at the start)
- `model.print_trainable_parameters()` prints the number of trainable parameters, and the total

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from peft import LoraConfig, get_peft_model

RANK = ...


def adapter_size(d_in: int, d_out: int) -> int:
  return ...


cfg = model.config
d = cfg.hidden_size
d_q = ...
d_kv = ...
per_layer = (
  adapter_size(d, d_q)  # q_proj
  + ...  # k_proj and v_proj
  + ...  # o_proj
)
print(f"{per_layer:,} per layer, {per_layer * cfg.num_hidden_layers:,} in all")


def lora(target_modules: list[str] | str) -> LoraConfig:
  return LoraConfig(
    r=RANK,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=target_modules,
    task_type="CAUSAL_LM",
  )


model = get_peft_model(..., lora(["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

# Another copy: get_peft_model changes the model it receives.
other = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map=DEVICE)
get_peft_model(other, lora(...)).print_trainable_parameters()
del other
```

### Solution

In [ ]:
from peft import LoraConfig, get_peft_model

RANK = 16


def adapter_size(d_in: int, d_out: int) -> int:
  return RANK * (d_in + d_out)


cfg = model.config
d = cfg.hidden_size
d_q = cfg.num_attention_heads * cfg.head_dim
d_kv = cfg.num_key_value_heads * cfg.head_dim
per_layer = (
  adapter_size(d, d_q)  # q_proj
  + 2 * adapter_size(d, d_kv)  # k_proj and v_proj
  + adapter_size(d_q, d)  # o_proj
)
print(f"{per_layer:,} per layer, {per_layer * cfg.num_hidden_layers:,} in all")


def lora(target_modules: list[str] | str) -> LoraConfig:
  return LoraConfig(
    r=RANK,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=target_modules,
    task_type="CAUSAL_LM",
  )


model = get_peft_model(model, lora(["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()

# Another copy: get_peft_model changes the model it receives.
other = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE, device_map=DEVICE)
get_peft_model(other, lora("all-linear")).print_trainable_parameters()
del other

# The computation is right: 4,587,520 parameters, 0.76% of the model. On all the linear
# layers, 10,092,544, or 1.7%: 2.2 times more, and still far from full fine-tuning's 600
# million.

## 4. Train

Train one epoch with `trl.SFTTrainer` and `SFTConfig`. The loss is computed on the answer (`completion`) only, not on the prompt. Plot the training and validation loss.

An epoch takes about 8 minutes on a GPU, hours without one: `train_set` and `eval_set` then hold 2 examples each, to see the loop run (about a minute), and the next exercise loads the adapters trained on a GPU by the same code.

**Useful documentation**

- [`SFTTrainer`](https://huggingface.co/docs/trl/sft_trainer)
- [`SFTConfig`](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)
- [`TrainerState`](https://huggingface.co/docs/transformers/main_classes/callback#transformers.TrainerState)

In [ ]:
# Everything on a GPU; without one, 2 training and 2 validation examples.
train_set = split["train"] if GPU else split["train"].select(range(2))
eval_set = split["test"] if GPU else split["test"].select(range(2))

# Your code here

### Additional hints

The calls and parameters to use:

- `SFTConfig(output_dir="lora", num_train_epochs=1, per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=2e-4, fp16=GPU, bf16=False, logging_steps=0.1, eval_strategy="steps", eval_steps=0.25, max_length=1024, report_to="none")`; `fp16=GPU`: half precision, on a GPU only; `bf16=False`: without it, TRL would compute in bfloat16 when `fp16` is false, which a CPU can't; `logging_steps=0.1`, `eval_steps=0.25`: a number below 1 is a share of the training steps (a measure every tenth, a validation every quarter)
- `SFTTrainer(model=model, args=args, train_dataset=train_set, eval_dataset=eval_set, processing_class=tokenizer)`: on a *prompt-completion* dataset, the loss only covers the answer
- `trainer.state.log_history`: the list of logged measures; `pd.DataFrame(...)` turns it into a table (columns `step`, `loss`, `eval_loss`), each row of which has only one of the two losses: `dropna(subset=["loss"])` keeps the training ones, `dropna(subset=["eval_loss"])` the validation ones

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import matplotlib.pyplot as plt
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
  output_dir="lora",
  num_train_epochs=1,
  per_device_train_batch_size=8,
  gradient_accumulation_steps=2,
  learning_rate=2e-4,
  fp16=GPU,
  bf16=False,
  logging_steps=0.1,
  eval_strategy="steps",
  eval_steps=0.25,
  max_length=1024,
  report_to="none",
)
trainer = SFTTrainer(
  model=model,
  args=args,
  train_dataset=...,
  eval_dataset=...,
  processing_class=tokenizer,
)
start = time.time()
trainer.train()
print(f"{(time.time() - start) / 60:.1f} min of training")

history = pd.DataFrame(trainer.state.log_history)
plt.plot(
  history.dropna(subset=["loss"]).step,
  history.dropna(subset=["loss"]).loss,
  label="training",
)
plt.plot(
  history.dropna(subset=[...]).step,
  history.dropna(subset=[...]).eval_loss,
  "o-",
  label="validation",
)
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()
```

### Solution

In [ ]:
import matplotlib.pyplot as plt
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
  output_dir="lora",
  num_train_epochs=1,
  per_device_train_batch_size=8,
  gradient_accumulation_steps=2,
  learning_rate=2e-4,
  fp16=GPU,
  bf16=False,
  logging_steps=0.1,
  eval_strategy="steps",
  eval_steps=0.25,
  max_length=1024,
  report_to="none",
)
trainer = SFTTrainer(
  model=model,
  args=args,
  train_dataset=train_set,
  eval_dataset=eval_set,
  processing_class=tokenizer,
)
start = time.time()
trainer.train()
print(f"{(time.time() - start) / 60:.1f} min of training")

history = pd.DataFrame(trainer.state.log_history)
plt.plot(
  history.dropna(subset=["loss"]).step,
  history.dropna(subset=["loss"]).loss,
  label="training",
)
plt.plot(
  history.dropna(subset=["eval_loss"]).step,
  history.dropna(subset=["eval_loss"]).eval_loss,
  "o-",
  label="validation",
)
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

## 5. The Fine-Tuned Small Model against the Large One

Rerun the extraction of the passages of `gold_part` with the fine-tuned model (without a GPU, the exercise's cell first loads the adapters trained on a GPU). Compare, in one table, the small model's F1 before and after fine-tuning and the large model's, with the time per passage. Did fine-tuning close the gap? What do we gain by putting the small model in production? The 40 test passages (`gold.jsonl`) were annotated by another LLM (Claude), each entity checked against the text: a lab shortcut, when the rule calls for a test set annotated by humans.

**Useful documentation**

- [`LoraModel.merge_and_unload`](https://huggingface.co/docs/peft/package_reference/tuners#peft.tuners.tuners_utils.BaseTuner.merge_and_unload)

In [ ]:
# Without a GPU, the adapters saw 2 examples only: we load those trained on a GPU by the
# same code, one epoch over the 1,899 examples.
if not GPU:
  from peft import set_peft_model_state_dict
  from safetensors.torch import load_file

  set_peft_model_state_dict(model, load_file("adapter_model.safetensors"))

# Your code here

### Additional hints

The calls and parameters to use:

- `model.merge_and_unload()` merges the adapters into the model's weights (no separate LoRA computation at each layer); `model.eval()` afterwards
- `extract_all(gold_part.text.tolist())`: exercise 1's extraction, with the fine-tuned model; the time per passage is measured as there
- `pd.DataFrame({name: evaluate(predictions), ...})`: one column per model; the large model, on the same passages: `evaluate(teacher[: len(gold_part)].to_dict("records"))`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
model = model.merge_and_unload()
model.eval()
start = time.time()
tuned_predictions, invalid = extract_all(...)
tuned_seconds = (time.time() - start) / len(gold_part)
print(f"{invalid} invalid answers, {tuned_seconds:.2f} s per passage")

pd.DataFrame(
  {
    "Qwen3-0.6B": evaluate(base_predictions),
    "Qwen3-0.6B + LoRA": evaluate(...),
    "large model (4B)": evaluate(...),
  }
)
```

### Solution

In [ ]:
# The adapters merged into the weights: no separate LoRA computation at each layer.
model = model.merge_and_unload()
model.eval()
start = time.time()
tuned_predictions, invalid = extract_all(gold_part.text.tolist())
tuned_seconds = (time.time() - start) / len(gold_part)
print(f"{invalid} invalid answers, {tuned_seconds:.2f} s per passage")

pd.DataFrame(
  {
    "Qwen3-0.6B": evaluate(base_predictions),
    "Qwen3-0.6B + LoRA": evaluate(tuned_predictions),
    "large model (4B)": evaluate(teacher[: len(gold_part)].to_dict("records")),
  }
)

# Measured on a GPU on November 4, 2026, with the 40 passages: fine-tuning closes, on
# this narrow task, most of the gap with the large model on people and places, less on
# organizations. Without a GPU, the 4 passages point the same way, but that is too few
# to conclude. The model is six times smaller: in production, it fits a much cheaper
# machine, or handles more passages at once on the same GPU. The large model's labels
# aren't perfect (a person filed under organizations...): the student learns its
# mistakes too.
# Fine-tuning costs a little reliability: in the same run, 3 invalid answers out of 40,
# against 0 for the base model.